In [1]:
from SelfCal import PipelineWrapper
from astropy.io import fits
import numpy as np
import glob
from SelfCal.SPHERExUtility import interpolate_array, make_chunk_map, make_chunk_mask, visualize_chunk_map, interp_2d_vertical, load_calibration
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 400 # User can set this outside the class if needed
# Import LogNorm
from matplotlib.colors import LogNorm
from tqdm import tqdm
import os
from SelfCal import MakeMap
import gc
from IPython.display import clear_output
import importlib


### Load Exposures

In [ ]:
detector = 4
# exposure_list = glob.glob(f'/data1/SPHEREx/reproc_data/deep_north/*/*/*/*D{detector}*.fits')
exposure_list = glob.glob(f'/data1/SPHEREx/galactic_plane/*/*/*/*D{detector}*.fits')
for exp_file in exposure_list:
    hdul = fits.open(exp_file)
    header = hdul[1].header
    good_astrometry = header.get('FINAST', 2)
    if good_astrometry != 0:
        print(f"Skipping {exp_file} due to poor astrometry (FINAST={good_astrometry})")
        exposure_list.remove(exp_file)
print(f"Found {len(exposure_list)} exposures")

Skipping /data1/SPHEREx/galactic_plane/2025W20_2D/l2b-v12-2025-175/4/level2_2025W20_2D_0444_1D4_spx_l2b-v12-2025-175.fits due to poor astrometry (FINAST=2)
Skipping /data1/SPHEREx/galactic_plane/2025W20_2D/l2b-v12-2025-175/4/level2_2025W20_2D_0554_3D4_spx_l2b-v12-2025-175.fits due to poor astrometry (FINAST=2)
Skipping /data1/SPHEREx/galactic_plane/2025W20_2D/l2b-v12-2025-175/4/level2_2025W20_2D_0554_2D4_spx_l2b-v12-2025-175.fits due to poor astrometry (FINAST=2)
Skipping /data1/SPHEREx/galactic_plane/2025W22_1B/l2b-v12-2025-176/4/level2_2025W22_1B_0514_2D4_spx_l2b-v12-2025-176.fits due to poor astrometry (FINAST=2)
Skipping /data1/SPHEREx/galactic_plane/2025W22_1B/l2b-v12-2025-176/4/level2_2025W22_1B_0079_1D4_spx_l2b-v12-2025-176.fits due to poor astrometry (FINAST=1)
Skipping /data1/SPHEREx/galactic_plane/2025W22_1B/l2b-v12-2025-176/4/level2_2025W22_1B_0033_1D4_spx_l2b-v12-2025-176.fits due to poor astrometry (FINAST=2)
Skipping /data1/SPHEREx/galactic_plane/2025W21_1B/l2b-v12-2025-1

### Reproject to mosaic frame

In [ ]:
config = {}
config['output_dir'] = '/data1/thomasli/selfcal/outputs'
config['run_name'] = f'gp_det{detector}_6p2arcsec'
config['resolution_arcsec'] = 6.2
# config['ref_path'] = '/home/thomasli/spherex/selfcal/outputs/common_ref.fits'

rr = PipelineWrapper.Reprojector(config, exposure_list=exposure_list)
rr.define_reference(padding_pixels=100, use_ext=[1])

NameError: name 'exposure_list' is not defined

In [ ]:
rr.run_reproject(max_workers=50, method='exact', padding_percentage=0.05, oversample_factor=2, 
                      sci_ext_list=[1], 
                      dq_ext_list=[2], 
                      exp_idx_list=np.arange(0, len(exposure_list)), 
                      det_idx_list=[0]*len(exposure_list),
                      replace_existing=False)

Starting batch reprojection. Output will be saved to: /home/thomasli/spherex/selfcal/outputs/gp_det4_6p2arcsec/reprojected




Reprojecting frames:   0%|                                                                                                                                                      | 0/189 [00:00<?, ?it/s]

In [ ]:
# Used for detecting and removing failed reprojections

# reproj_list = sorted(glob.glob('outputs/nep_det2_6p2arcsec/reprojected/*.h5'))

# det_idx_list = []
# exp_idx_list = []
# for file in tqdm(reproj_list):
#     file_name = os.path.basename(file)
#     det_idx_list.append(int(file_name[file_name.find('det_')+4:file_name.find('det_')+6]))
#     exp_idx_list.append(int(file_name[file_name.find('exp_')+4:file_name.find('exp_')+8]))

# for f in tqdm(reproj_list):
#     result = MakeMap.load_reproj_file(f, fields=['sub_data',])
#     if result['_is_missing_']:
#         os.remove(f)
#         print(f"Removed {f} due to missing data")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3288/3288 [00:00<00:00, 1188560.85it/s]


In [ ]:
# Used to visualize reprojections

# sample = MakeMap.load_reproj_file(reproj_list[0], fields=['sub_data', 'grid_bitmask'])
# mask = MakeMap.grid_bitmask_to_sub_mask(sample['grid_bitmask'], oversample_factor=2, ignore_list=[21])
# plt.imshow(sample['sub_data']*mask, origin='lower', norm=LogNorm())

### Solve for calibration

In [ ]:
chunk_map = make_chunk_map(detector, interp_factor=10)
chunk_valid_mask = make_chunk_mask([7, 8], interp_factor=10)
det_valid_mask = chunk_valid_mask[chunk_map]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172/172 [00:10<00:00, 15.70it/s]


In [ ]:
detector = 2
config = {}
config['output_dir'] = '/home/thomasli/spherex/selfcal/outputs'
config['run_name'] = f'gp_det{detector}_6p2arcsec'
config['resolution_arcsec'] = 6.2

cc = PipelineWrapper.Calibrator(config)

cc.setup_lsqr(
    apply_mask=True, 
    apply_weight=False, 
    chunk_map=chunk_map, 
    det_valid_mask=det_valid_mask, 
    max_workers=40, 
    outlier_thresh=100.0,
    ignore_list=[21],
    )

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 189/189 [00:00<00:00, 1111814.10it/s]

Loading reference frame from: /home/thomasli/spherex/selfcal/outputs/gp_det4_6p2arcsec/ref.fits



Building A, b matrix: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 189/189 [00:42<00:00,  4.47it/s]


In [ ]:
cc.apply_lsqr(x0=None, atol=1e-06, btol=1e-06, damp=1e-1, iter_lim=500)

cal_path = cc.save_calibration(cal_file='cal_det4_ch7-8.h5')

Solving least squares for 139877978 unknowns with 85432617 equations.
 
LSQR            Least-squares solution of  Ax = b
The matrix A has 85432617 rows and 139877978 columns
damp = 1.00000000000000e-01   calc_var =        0
atol = 1.00e-06                 conlim = 1.00e+08
btol = 1.00e-06               iter_lim =      500
 
   Itn      x[0]       r1norm     r2norm   Compatible    LS      Norm A   Cond A
     0  0.00000e+00   1.105e+04  1.105e+04    1.0e+00  7.1e-03
     1  0.00000e+00   9.419e+03  9.419e+03    8.5e-01  2.1e-02   1.5e+02  1.0e+00
     2  0.00000e+00   9.416e+03  9.416e+03    8.5e-01  8.9e-03   2.0e+02  2.0e+00
     3  0.00000e+00   9.373e+03  9.373e+03    8.5e-01  6.5e-02   2.1e+02  1.3e+01
     4  0.00000e+00   6.913e+03  6.913e+03    6.3e-01  3.3e-01   2.6e+02  1.1e+02
     5  0.00000e+00   4.488e+03  4.489e+03    4.1e-01  5.8e-02   2.9e+02  1.7e+02
     6  0.00000e+00   4.439e+03  4.456e+03    4.0e-01  6.7e-03   3.3e+02  1.9e+02
     7  0.00000e+00   4.435e+03  4.45

### Coadd with calibration

In [ ]:
mm = PipelineWrapper.Mosaicker(config)
mm.load_calibration(cal_path=cal_path)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 189/189 [00:00<00:00, 1102536.10it/s]

Loading reference frame from: /home/thomasli/spherex/selfcal/outputs/gp_det4_6p2arcsec/ref.fits


Calibration loaded from /home/thomasli/spherex/selfcal/outputs/gp_det4_6p2arcsec/calibration/cal_det4_ch7-8.h5


In [ ]:
maps = mm.make_mosaic(
    apply_mask=True, 
    apply_weight=False, 
    chunk_map=chunk_map, 
    det_valid_mask=det_valid_mask, 
    max_workers=20,
    make_std_map=True, 
    apply_sigma_clipping=True, 
    sigma=1.0,
    ignore_list=[21]
)

mm.save_mosaic(mos_file='gp_6p2arcsec_det4_ch7-8.fits', overwrite=True)

Computing sigma_clip map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19/19 [02:00<00:00,  6.34s/it]


In [ ]:
def plot_map(map, wcs=None, lowp=0.1, highp=95, ax=None, colorbar=True):
    high, low = np.nanpercentile(map[map>0], [highp, lowp])
    fig = plt.figure(figsize=(10, 10))
    if ax is None:
        ax = fig.add_subplot(111, projection=wcs)
    im = ax.imshow(map, norm=LogNorm(vmin=low, vmax=high), origin='lower')

    if wcs is not None:
        # Explicitly set axis labels
        ax.coords['ra'].set_axislabel('RA')
        ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
        ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
        ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
        ax.coords['dec'].set_axislabel('DEC')
        ax.grid(color='black', linestyle='--', alpha=0.5)

    if colorbar:
        cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
        cbar.set_label('MJy/sr')
    return fig, ax

In [ ]:
# plot_map(mm.maps['sc_mean_map'], wcs=mm.ref_wcs, lowp=0.1, highp=99.9)

### Batch processing example

In [ ]:
detector = 4
config = {}
config['output_dir'] = '/home/thomasli/spherex/selfcal/outputs'
config['run_name'] = f'nep_det{detector}_6p2arcsec'
config['resolution_arcsec'] = 6.2

In [ ]:
chunk_map = make_chunk_map(detector, interp_factor=10)

chs = [[2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17]]

for ch in chs:
    print(f"Processing channel {ch} for detector {detector}")
    chunk_valid_mask = make_chunk_mask(ch, interp_factor=10)
    det_valid_mask = chunk_valid_mask[chunk_map]
    cc = PipelineWrapper.Calibrator(config)
    cc.setup_lsqr(
        apply_mask=True, 
        apply_weight=False, 
        chunk_map=chunk_map, 
        det_valid_mask=det_valid_mask, 
        max_workers=40, 
        outlier_thresh=20.0,
        ignore_list=[],
        )

    cc.apply_lsqr(x0=None, atol=1e-06, btol=1e-06, damp=1e-1, iter_lim=500)

    cal_path = cc.save_calibration(cal_file=f'cal_det{detector}_ch{'-'.join(map(str, ch))}.h5')

    mm = PipelineWrapper.Mosaicker(config)
    
    mm.load_calibration(cal_path=cal_path)

    maps = mm.make_mosaic(
        apply_mask=True, 
        apply_weight=False, 
        chunk_map=chunk_map, 
        det_valid_mask=det_valid_mask, 
        max_workers=40,
        make_std_map=True, 
        apply_sigma_clipping=True, 
        sigma=1.0,
        ignore_list=[21],
        interp_func=interp_2d_vertical
    )

    mm.save_mosaic(mos_file=f'mosaic_det{detector}_ch{"-".join(map(str, ch))}.fits', overwrite=True)

    # Clear memory
    del cc, mm, maps
    gc.collect()
    # Clear jupyter output
    clear_output(wait=True)


Processing channel [6] for detector 4


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3454/3454 [00:00<00:00, 1402025.16it/s]

Loading reference frame from: /home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/ref.fits



Building A, b matrix: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3454/3454 [07:36<00:00,  7.56it/s]


Solving least squares for 81353914 unknowns with 299959310 equations.
 
LSQR            Least-squares solution of  Ax = b
The matrix A has 299959310 rows and 81353914 columns
damp = 1.00000000000000e-01   calc_var =        0
atol = 1.00e-06                 conlim = 1.00e+08
btol = 1.00e-06               iter_lim =      500
 
   Itn      x[0]       r1norm     r2norm   Compatible    LS      Norm A   Cond A
     0  0.00000e+00   1.292e+03  1.292e+03    1.0e+00  7.3e-02
     1  0.00000e+00   2.370e+02  2.370e+02    1.8e-01  7.3e-01   9.5e+01  1.0e+00
     2  0.00000e+00   1.460e+02  1.460e+02    1.1e-01  2.3e-01   1.3e+02  2.0e+00
     3  0.00000e+00   1.363e+02  1.363e+02    1.1e-01  7.0e-02   1.6e+02  3.1e+00
     4  0.00000e+00   1.351e+02  1.351e+02    1.0e-01  2.5e-02   1.8e+02  4.3e+00
     5  0.00000e+00   1.348e+02  1.348e+02    1.0e-01  1.7e-02   1.9e+02  5.5e+00
     6  0.00000e+00   1.344e+02  1.344e+02    1.0e-01  2.7e-02   2.1e+02  8.4e+00
     7  0.00000e+00   1.328e+02  1.32

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3454/3454 [00:00<00:00, 1425757.90it/s]

Loading reference frame from: /home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/ref.fits


Calibration loaded from /home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/calibration/cal_det4_ch6.h5


Computing mean map:   0%|                                                                                                                                                        | 0/40 [00:00<?, ?it/s]

### Append Wavelength

In [ ]:
det_BC, det_BW = load_calibration(band=1, calibration_dir='/data1/SPHEREx/reproc_data/calibrations')

chunk_map = make_chunk_map(1, det_BC, interp_factor=1,
                   channel_file='/home/thomasli/spherex/spherex_nep_catalogues/spherex_channels.csv')

chunk_valid_mask = make_chunk_mask([1], interp_factor=1)
det_valid_mask = chunk_valid_mask[chunk_map]

det_stack = np.stack([det_valid_mask, det_BC, det_BW])

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19/19 [00:01<00:00,  9.90it/s]


In [ ]:
mos_hdul = fits.open('/data1/thomasli/selfcal/mosaics/mosaic_det1_ch1.fits')

mean_map = mos_hdul[1].data
std_map = mos_hdul[3].data
ref_shape = mean_map.shape

BCBW_sum = np.zeros_like(mean_map)
BW_sum = np.zeros_like(mean_map)
meanvar_sum = np.zeros_like(mean_map)

In [ ]:
from SelfCal.MakeMap import det_to_grid, bin2d, compute_crop
reproj_list = sorted(glob.glob('/data1/thomasli/selfcal/outputs/nep_det1_6p2arcsec/reprojected/*.h5'))


In [ ]:
for file in tqdm(reproj_list[0:100]):

    result = MakeMap.load_reproj_file(file, fields=['ref_coords', 'grid_mapping', 'sub_data'])
    grid_mapping = result['grid_mapping']
    ref_coords = result['ref_coords']
    sub_data = result['sub_data']

    grid_stack = det_to_grid(grid_mapping, det_stack )
    sub_stack = bin2d(grid_stack, 2, bin_func=np.mean)
    sub_mask, sub_BC, sub_BW = sub_stack
    sub_mask = sub_mask == 1

    sub_crop, ref_crop = compute_crop(ref_shape, ref_coords)

    mean_crop = mean_map[ref_crop]
    std_crop = std_map[ref_crop]
    data_crop = sub_data[sub_crop]
    sigma = 1
    clip_mask = np.abs(data_crop - mean_crop) <= sigma * std_crop
    crop_mask = sub_mask[sub_crop] & clip_mask

    BCBW_sum[ref_crop] += crop_mask * (sub_BC * sub_BW)[sub_crop]
    BW_sum[ref_crop] += crop_mask * sub_BW[sub_crop]
    meanvar_sum[ref_crop] += crop_mask * (sub_BW * ((sub_BW**2)/12 + sub_BC**2))[sub_crop]


 12%|██████████████                                                                                                       | 12/100 [00:36<04:25,  3.02s/it]


KeyboardInterrupt: 

In [ ]:
from functools import partial
import multiprocessing as mp


def _process_batch(file_batch, det_stack, ref_shape, mean_map, std_map):
    """Internal worker function to process a single batch of files."""
    local_BCBW_sum = np.zeros(ref_shape, dtype=np.float64)
    local_BW_sum = np.zeros(ref_shape, dtype=np.float64)
    local_meanvar_sum = np.zeros(ref_shape, dtype=np.float64)

    for file in file_batch:
        result = MakeMap.load_reproj_file(file, fields=['ref_coords', 'grid_mapping', 'sub_data'])
        grid_mapping, ref_coords, sub_data = result['grid_mapping'], result['ref_coords'], result['sub_data']

        grid_stack = det_to_grid(grid_mapping, det_stack)
        sub_mask, sub_BC, sub_BW = bin2d(grid_stack, 2, bin_func=np.mean)
        sub_mask = sub_mask == 1

        sub_crop, ref_crop = compute_crop(ref_shape, ref_coords)

        mean_crop = mean_map[ref_crop]
        std_crop = std_map[ref_crop]
        data_crop = sub_data[sub_crop]
        sigma = 1
        clip_mask = np.abs(data_crop - mean_crop) <= sigma * std_crop
        crop_mask = sub_mask[sub_crop] & clip_mask

        local_BCBW_sum[ref_crop] += crop_mask * (sub_BC * sub_BW)[sub_crop]
        local_BW_sum[ref_crop] += crop_mask * sub_BW[sub_crop]
        local_meanvar_sum[ref_crop] += crop_mask * (sub_BW * ((sub_BW**2) / 12 + sub_BC**2))[sub_crop]

    return local_BCBW_sum, local_BW_sum, local_meanvar_sum


num_cores = 40
file_batches = np.array_split(reproj_list, num_cores)

worker_func = partial(_process_batch,
                        det_stack=det_stack,
                        ref_shape=ref_shape,
                        mean_map=mean_map,
                        std_map=std_map)

with mp.Pool(processes=num_cores) as pool:
    results = list(tqdm(pool.map(worker_func, file_batches), 
                        total=len(file_batches), 
                        desc="Processing Batches"))

# Reduce the results from all batches into the final arrays
BCBW_sum = sum(item[0] for item in results)
BW_sum = sum(item[1] for item in results)
meanvar_sum = sum(item[2] for item in results)

: 

: 

: 

In [ ]:
BC_mean = BCBW_sum / BW_sum
BC_std = np.sqrt(meanvar_sum/BW_sum - BC_mean**2)

/tmp/ipykernel_17388/772798825.py:1: RuntimeWarning: invalid value encountered in divide
  BC_mean = BCBW_sum / BW_sum
/tmp/ipykernel_17388/772798825.py:2: RuntimeWarning: invalid value encountered in divide
  BC_std = np.sqrt(meanvar_sum/BW_sum - BC_mean**2)


: 

In [ ]:
np.nanmean(BC_std)

np.float64(0.006741847870063144)

In [ ]:
np.nanmax(BC_mean)-np.nanmin(BC_mean)

np.float64(0.018444598562478265)

In [ ]:
im = plt.imshow(mos_hdul[5].data, norm=LogNorm())
plt.colorbar(im)

NameError: name 'plt' is not defined